In [ ]:
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"
unit_table_opto_metrics = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_opto_metrics.csv"
unit_cluster_labels = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_cluster_labels.csv"
unit_vis_responsive = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_vis_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv"

In [ ]:
units = pd.read_csv(unit_table_file)

### Generated on HPC by 'run_image_decoding_sliding_window.py'

In [ ]:
decoding_results_files = os.listdir("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding_sliding_window")
decoding_results_files = [os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding_sliding_window", d) for d in decoding_results_files]

In [ ]:
sessions_table = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.4.0/project_metadata/ecephys_sessions.csv")
no_anomalies = sessions_table[(sessions_table['abnormal_activity'].isnull())&(sessions_table['abnormal_histology'].isnull())]

In [ ]:
from notebook_utils import (
    get_decoding_results_files, get_mouse_from_session, get_mouse_paired_indices
)

In [ ]:
conditions = ['Familiar', 'Novel']

In [ ]:
session_decoding_files = get_decoding_results_files(no_anomalies['ecephys_session_id'].values, decoding_results_files, image_set='changeRS')
decoding_results_dict = {}
for decoding_file in session_decoding_files:

    d = np.load(decoding_file, allow_pickle=True).item()
    decoding_results_dict[decoding_file] = {'VISall': d['VISall']}

decode_windows = d['decodeWindows']

In [ ]:
# sample_size = 20
# region = 'VISp'
sample_size = 80
region = 'VISall'

session_feature_weights = {}
session_cluster_weights = {}
for idf, decoding_file in enumerate(session_decoding_files):
    if idf % 5 == 0:
        print(f"Processing decoding file {idf+1}/{len(session_decoding_files)}")

    d = decoding_results_dict[decoding_file]
    
    if region not in d:
        continue

    if sample_size not in d[region]:
        continue

    if 'featureWeights' not in d[region][sample_size]:
        continue

    if 'unit_ids' not in d[region][sample_size]:
        continue

    session_id = os.path.basename(decoding_file).split('_')[0]
    data = np.array(d[region][sample_size]['featureWeights'])
    num_decode_windows = data.shape[0]
    num_unit_samples = data.shape[1]

    unit_ids = np.array(d[region][sample_size]['unit_ids'])
    unique_units = list(np.unique(unit_ids))
    feature_weights = np.full((num_decode_windows, num_unit_samples, 8, len(unique_units)), np.nan)
    cluster_weights_over_time = {c:{win: [] for win in np.arange(num_decode_windows)} for c in np.arange(13)}
    for window in range(num_decode_windows):
        for sampleind in range(num_unit_samples):

            us = unit_ids[window, sampleind]
            ws_data = data[window][sampleind].reshape((8, sample_size, -1)).mean(axis=2)

            for uind, u in enumerate(us):
                u_index = unique_units.index(u)
                u_weights = ws_data[:, uind]
                feature_weights[window, sampleind, :, u_index] = u_weights

                ucluster = units[units['unit_id']==u]['cluster_labels_new'].values[0]
                if np.isnan(ucluster):
                    ucluster = 0
                cluster_weights_over_time[int(ucluster)][window].append(np.nanmean(u_weights))

    session_feature_weights[session_id] = feature_weights
    session_cluster_weights[session_id] = cluster_weights_over_time

In [ ]:
session_correlation_mats = []
for session_id, feature_weights in session_feature_weights.items():
    fws = np.nanmean(feature_weights, axis=1)
    fwc = np.corrcoef(np.nanmean(fws,axis=1))
    session_correlation_mats.append(fwc)


plt.imshow(np.nanmean(session_correlation_mats, axis=0))
plt.colorbar()

In [ ]:
across_session_cluster_weights = {c:[] for c in np.arange(6)}
for session_id, cluster_weights in session_cluster_weights.items():
    for cluster in np.arange(6):
        over_time = [np.mean(np.abs(cluster_weights[cluster][w])) for w in np.arange(num_decode_windows)]
        if len(over_time) == 0:
            continue
        across_session_cluster_weights[cluster].append(over_time)

In [ ]:
fig, ax = plt.subplots()
for cluster in np.arange(6):
    ax.plot(decode_windows, np.nanmean(across_session_cluster_weights[cluster], axis=0), label=f'Cluster {cluster}')
ax.set_ylabel('Decoder weight')
ax.set_xlabel('End of 50 ms sliding decoding window (ms)')
plt.legend()

In [ ]:
for cluster in np.arange(1,7):
    plt.plot([np.mean(np.abs(cluster_weights_over_time[cluster][w])) for w in np.arange(num_decode_windows)], label=str(cluster))

plt.legend()

In [ ]:
early = np.nanmean(fws[2:12], axis=0)
high_val_unit_inds = [uind for uind, uval in enumerate(early.T) if any(uval>0.5)]
early_units = np.array(unique_units)[high_val_unit_inds]
early_units

In [ ]:
plt.figure()
plt.imshow(fws[-2])
plt.figure()
plt.imshow(fws[-1])

In [ ]:
fwc = np.corrcoef([f.flatten() for f in fws])

In [ ]:
fwc = np.corrcoef(np.nanmean(fws,axis=1))

In [ ]:
plt.imshow(fwc)

In [ ]:
regions = ('VISall','VISp','VISl','VISrl','VISal','VISpm','VISam','LP', 'LGd',
            'SCMRN', 'Hipp')
unit_sub_sample_sizes = list(d['VISall'].keys())
region_timecourses = {}
session_ids = {r:{c: {n:[] for n in unit_sub_sample_sizes} for c in conditions} for r in regions}
for region in regions:
    imagewise_recall_per_condition = {c:{} for c in conditions}
    for condition in conditions:

        condition_sessions = no_anomalies[no_anomalies['experience_level']==condition]
        condition_decoding_files = get_decoding_results_files(condition_sessions['ecephys_session_id'].values, decoding_results_files, image_set='changeRS')
        
        for n in unit_sub_sample_sizes:
            bas = []
            sess_ids = []
            for cdf in condition_decoding_files:
                d = decoding_results_dict[cdf]
                #d = np.load(cdf, allow_pickle=True).item()
                # ba = np.array(d[region][n]['imagewise_precision'])
                ba = np.array(d[region][n]['balanced_accuracy'])
                if ba.size>0:
                    bas.append(ba)
                    sess_ids.append(int(os.path.basename(cdf).split('_')[0]))

            #ba_array = np.array([a for a in bas if a.size>0])
            ba_array = np.array(bas)
            imagewise_recall_per_condition[condition][n] = ba_array
            session_ids[region][condition][n] = sess_ids
        # fig, ax = plt.subplots()
        # ax.plot(np.mean(imagewise_recall_per_condition['Novel'], axis=0))
        # ax.legend(np.arange(8))
        # ax.set_xlim([0,15])
        # fig, ax = plt.subplots()
        # ax.plot(np.mean(imagewise_recall_per_condition['Familiar'], axis=0))
        # ax.legend(np.arange(8))
        # ax.set_xlim([0,15])
    region_timecourses[region] = imagewise_recall_per_condition

In [ ]:
region_timecourses['VISp']['Familiar'][20].shape

In [ ]:
import vbn_utils

In [ ]:
region_timecourses['VISp']['Novel'][20].shape

In [ ]:
vis_all = np.concatenate([region_timecourses['VISall']['Familiar'][80], region_timecourses['VISall']['Novel'][80]], axis=0)
vis_all.shape

In [ ]:
fig, ax = plt.subplots()
# vbn_utils.mean_sem_plot(np.mean(region_timecourses['VISall']['Familiar'][80], axis=2), ax=ax, color='b', x=decode_windows-25)
# vbn_utils.mean_sem_plot(np.mean(region_timecourses['VISall']['Novel'][80], axis=2), ax=ax, color='r', x=decode_windows-25)
# vbn_utils.mean_sem_plot(region_timecourses['VISall']['Familiar'][80], ax=ax, color='b', x=decode_windows-25)
# vbn_utils.mean_sem_plot(region_timecourses['VISall']['Novel'][80], ax=ax, color='r', x=decode_windows-25)
vbn_utils.mean_sem_plot(vis_all, ax=ax, color='k', x=decode_windows-25)
vbn_utils.formatFigure(fig, ax, xLabel='Time from stimulus onset (ms)', yLabel='Decoder accuracy')
ax.set_ylim([0, 1])
ax.axhline(0.125, color='k', linestyle='--', linewidth=0.5)
# ax.plot(np.mean(region_timecourses['VISall']['Familiar'][80], axis=(0,2)))

In [ ]:
from notebook_utils import (
    fitCurve,
    calc_gompertz,
    invert_gompertz,
    get_sigmoidfit_midpoint,
    get_sigmoidfit_midpoint_2val,
    find_midpoint_raw
)

In [ ]:
fits = []
rvals = []
avals = []
fitdatas = []
for s in region_timecourses['Hipp']['Novel'][20]:
    a = np.mean(s, axis=1)
    avals.append(a)
    #fig, ax = plt.subplots()
    #ax.plot(decode_windows, a)
    
    xmid, ymid = get_sigmoidfit_midpoint_2val(decode_windows, a)
    #xraw, yraw = find_midpoint_raw(decode_windows, a)
    # ax.plot(xmid, ymid, 'ro')
    # ax.plot(xraw, yraw, 'go')
    fit = fitCurve(calc_gompertz, decode_windows, a, [a.max(), 1, 5, a.min()])
    fits.append(fit)
    fit_data = [calc_gompertz(x, *fit) for x in decode_windows]
    fitdatas.append(fit_data)
    rval = np.corrcoef(fit_data,a)[0,1]
    rvals.append(rval)
    # ax.plot(decode_windows, fit_data)
    
    # print(fit)
    # mid_range_y = (fit[0]-fit[-1])/2 + fit[-1]

    # x50 = invert_gompertz(mid_range_y, np.arange(0, decode_windows.max(), 0.1),*fit)
    # ax.axvline(x50)
    # ax.axhline(mid_range_y)


In [ ]:
for region in regions:
    fig, axes = plt.subplots(1,len(unit_sub_sample_sizes))
    fig.set_size_inches(18,6)
    imagewise_recall_per_condition = region_timecourses[region]

    for n, ax in zip(unit_sub_sample_sizes, axes):
        colors = ['b', 'r']
        counts = []
        for ic, condition in enumerate(['Familiar', 'Novel']):
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                mean = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(0,2))
                std = np.std(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(0,2))
                sem = std/count**0.5
                time = decode_windows#np.arange(0, len(mean)*10, 10)
                ax.plot(time, mean, colors[ic])
                ax.fill_between(time, mean+sem, mean-sem, color=colors[ic], alpha=0.3)
        
        ax.set_title(f'{region} {n}, f: {counts[0]}, n: {counts[1]}')
        ax.set_xlim([0,750])
        ax.set_ylim([0,1])
        ax.axhline(0.125, color='k', ls='dotted')


In [ ]:
np.where(np.array(latencies['APN']['Novel'][20])>140)

In [ ]:
import scipy.signal
tc = np.mean(region_timecourses['APN']['Novel'][20][28, :, :6], axis=1)
tc_filt = scipy.signal.medfilt(tc, 3)
plt.plot(decode_windows, tc)
plt.plot(decode_windows, tc_filt)
print(get_sigmoidfit_midpoint(decode_windows, tc))
ymidpoint = (tc.max() - tc.min())/2 + tc.min()
shift_estimate = np.where(tc>ymidpoint)[0][0]
print(shift_estimate)
fit = fitCurve(calc_gompertz, decode_windows, tc, [tc.max(), 0.3, shift_estimate, tc.min()])
print(fit)
fit_data = [calc_gompertz(x, *fit) for x in decode_windows]
xraw, yraw = find_midpoint_raw(decode_windows, tc_filt)
plt.plot(xraw, yraw, 'go')
plt.plot(decode_windows, fit_data)

In [ ]:
#accuracy at full time an n=20
hit_rates = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
for region in regions:
    for n in unit_sub_sample_sizes[:-1]:
        for condition in conditions:
            timecourse = region_timecourses[region][condition][n]
            count = timecourse.shape[0]
            counts.append(count)
            if count>0:
                timecourse_means = np.mean(timecourse[:,:,:6], axis=2)
                hrs = timecourse_means[:, 15]
                hit_rates[region][condition][n] = hrs
                

In [ ]:
from notebook_utils import bh_multitest

In [ ]:
regions_to_plot = ['LGd', 'VISp', 'VISl', 'VISrl', 'LP', 'VISal', 'VISpm', 'VISam', 'Hipp', 'MRN']
plt.figure()
pvals = []
for ir, region in enumerate(regions_to_plot):#regions:

    vals = [np.array(hit_rates[region][cond][20]) for cond in ['Familiar', 'Novel']]
    pval = scipy.stats.mannwhitneyu(*vals, nan_policy='omit')[1]
    pvals.append(pval)
    # print(ir)
    # print(np.nanmean(vals[0]))
    plt.plot(ir, np.nanmean(vals[0]), 'bo')
    plt.plot(ir+0.1, np.nanmean(vals[1]), 'ro')

    plt.errorbar(ir, np.nanmean(vals[0]), np.nanstd(vals[0])/(np.sum(~np.isnan(vals[0]))**0.5), color='b')
    plt.errorbar(ir+0.1, np.nanmean(vals[1]), np.nanstd(vals[1])/(np.sum(~np.isnan(vals[1]))**0.5), color='r')

ax = plt.gca()
ax.set_xticks(np.arange(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot)
ax.set_ylabel('Decoding accuracy (ms)')

sig_after_correction = bh_multitest(pvals)[0]
sigx_ind = np.where(sig_after_correction)[0][0]
ax.text(sigx_ind, 1, '*')

In [ ]:
latencies = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
latency_sessions = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}

for region in ['LGd', 'VISp', 'VISl']:
    imagewise_recall_per_condition = region_timecourses[region]
    plt.figure()
    fig = plt.gcf()
    fig.set_size_inches([12,6])
    for n in [20]:
        for ic, condition in enumerate(['Familiar', 'Novel']):
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                image_means = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(2))
                lats = [get_sigmoidfit_midpoint(decode_windows, y)[0] for y in image_means]
                ylats = [get_sigmoidfit_midpoint(decode_windows, y)[1] for y in image_means]
                plt.plot(decode_windows, image_means.T, colors[ic], alpha=0.2)
                plt.plot(lats, ylats, colors[ic]+'o', alpha=0.5, mec='none')
                # sess_ids = [s for s,lat in zip(session_ids[region][condition][n], lats) if ~np.isnan(lat)]
                #lats = [find_midpoint_raw(decode_windows, y)[0] for y in image_means]
                latencies[region][condition][n] = lats
                # latency_sessions[region][condition][n] = sess_ids
                plt.plot(decode_windows, np.mean(image_means, axis=0), colors[ic], lw=2)
                print(f'{region}: {np.median(lats)}')
    ax = plt.gca()
    ax.set_xlim([0, 150])

In [ ]:
latencies = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}
latency_sessions = {r: {c:{n:[] for n in unit_sub_sample_sizes} for c in ['Familiar', 'Novel']} for r in regions}

for region in regions:
    imagewise_recall_per_condition = region_timecourses[region]

    for n in [10,20,40]:
        for ic, condition in enumerate(['Familiar', 'Novel']):
            count = imagewise_recall_per_condition[condition][n].shape[0]
            counts.append(count)
            if count>0:
                image_means = np.mean(imagewise_recall_per_condition[condition][n][:, :, :6], axis=(2))
                lats = [get_sigmoidfit_midpoint(decode_windows, y)[0] for y in image_means]
                # sess_ids = [s for s,lat in zip(session_ids[region][condition][n], lats) if ~np.isnan(lat)]
                #lats = [find_midpoint_raw(decode_windows, y)[0] for y in image_means]
                latencies[region][condition][n] = lats
                # latency_sessions[region][condition][n] = sess_ids

In [ ]:
import scipy.stats
for region in regions:
    
    vals = [np.array(latencies[region][cond][20]) for cond in ['Familiar', 'Novel']]
    vals = [v[~np.isnan(v)] for v in vals]
    counts = [len(v) for v in vals]
    medians = [np.median(v) for v in vals]
    fig, ax = plt.subplots()
    ax.violinplot(vals)
    print(f'{region}, f: {medians[0]}, n: {medians[1]}, diff: {medians[1]-medians[0]}')
    pval = scipy.stats.mannwhitneyu(*vals)[1]
    fig.suptitle(f'{region} p={pval} counts={counts[0]}, {counts[1]}')

In [ ]:
import scipy.stats
region_diffs = []
for region in ['LGd', 'LP', 'VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'VISall']:#regions:

    mouse_match_inds = get_mouse_paired_indices(session_ids, region, 20)
    vals = [np.array(latencies[region][cond][20]) for cond in ['Familiar', 'Novel']]

    mouse_matched = [v[mouse_match_inds[c]] for v,c in zip(vals, ['Familiar', 'Novel'])]

    diffs = mouse_matched[1] - mouse_matched[0]
    region_diffs.append(diffs)
    print(np.median(diffs))
    print(mouse_matched)

fig, ax = plt.subplots()
ax.violinplot(region_diffs)

In [ ]:
regions_to_plot = ['LGd', 'VISp', 'VISl', 'VISrl', 'LP', 'VISal', 'VISpm', 'VISam', 'Hipp', 'MRN']
region_means = []
plt.figure()
pvals = []
for ir, region in enumerate(regions_to_plot):#regions:

    vals = [np.array(latencies[region][cond][20]) for cond in ['Familiar', 'Novel']]
    pvals.append(scipy.stats.mannwhitneyu(*vals, nan_policy='omit')[1])
    # print(ir)
    # print(np.nanmean(vals[0]))
    plt.plot(ir, np.nanmean(vals[0]), 'bo')
    plt.plot(ir+0.1, np.nanmean(vals[1]), 'ro')

    plt.errorbar(ir, np.nanmean(vals[0]), np.nanstd(vals[0])/(np.sum(~np.isnan(vals[0]))**0.5), color='b')
    plt.errorbar(ir+0.1, np.nanmean(vals[1]), np.nanstd(vals[1])/(np.sum(~np.isnan(vals[1]))**0.5), color='r')

ax = plt.gca()
ax.set_xticks(np.arange(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot)
ax.set_ylabel('Decoding latency (ms)')

sig_after_correction = bh_multitest(pvals)[0]
sigx_ind = np.where(sig_after_correction)[0]
[ax.text(x, 150, '*') for x in sigx_ind]

In [ ]:
import ccf_utils
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/ccf_structure_tree_2017.csv")


In [ ]:
unit_sub_sample_sizes = [1, 5, 10, 20, 40, 60, 80]
total_decoding_windows = 30
subsample_balanced_accuracy = {cond: {n:[] for n in unit_sub_sample_sizes} for cond in ['Familiar', 'Novel']}
for condition in conditions:

    condition_sessions = no_anomalies[no_anomalies['experience_level']==condition]
    condition_decoding_files = get_decoding_results_files(condition_sessions['ecephys_session_id'].values, decoding_results_files, image_set='change')
    
    bas = []
    for cdf in condition_decoding_files:
        d = decoding_results_dict[cdf]
        #d = np.load(cdf, allow_pickle=True).item()
        for n in unit_sub_sample_sizes:
            ba = d['VISp'][n]['balanced_accuracy']
            if len(ba)<total_decoding_windows:
                continue
                #ba = np.full(total_decoding_windows, np.nan)
            #ba = np.array([d['VISp'][n]['balanced_accuracy'] for n in unit_sub_sample_sizes])

            subsample_balanced_accuracy[condition][n].append(ba)

In [ ]:
np.array(subsample_balanced_accuracy[condition][n]).shape[0]

In [ ]:
for condition in ['Familiar', 'Novel']:

    for n in unit_sub_sample_sizes:

        ba_array = np.array(subsample_balanced_accuracy[condition][n])
        num_exps = np.array(subsample_balanced_accuracy[condition][n]).shape[0]
        if num_exps>0:

            mean = np.mean(ba_array, axis=0)



In [ ]:
time_point = 5
fig, ax = plt.subplots()
for n in unit_sub_sample_sizes:

    for condition in ['Familiar', 'Novel']:

        ba_array = np.array(subsample_balanced_accuracy[condition][n])
        non_nan_exps = np.unique(np.where(~np.isnan(ba_array))[0])
        if len(non_nan_exps)>0:
            mean = np.mean(ba_array[non_nan_exps], axis=0)
            sem  = np.std(ba_array[non_nan_exps], axis=0)/(len(non_nan_exps))**0.5



In [ ]:
fig,ax = plt.subplots()
for ba_array in [subsample_balanced_accuracy['Familiar'], subsample_balanced_accuracy['Novel']]:
    decoding_curves = []
    for exp in ba_array:
        dcurve = []
        for n in range(len(unit_sub_sample_sizes)):
            vals = exp[n]
            if len(vals)==0:
                val = np.nan
            else:
                val = vals[5]
            dcurve.append(val)
        decoding_curves.append(dcurve)
    decoding_curve_array = np.array(decoding_curves).squeeze().shape

    ax.plot(unit_sub_sample_sizes, np.nanmean(decoding_curves, axis=0))
ax.set_xlabel('unit sample size')
ax.set_ylabel('decoder accuracy at full time')


In [ ]:
om = {}
for cluster in ['all', 1, 2, 3, 4, 5]:
    om[cluster] = np.load("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding_sliding_window/1091039902_throughomissionRS_" + str(cluster) + ".npy", allow_pickle=True).item()

In [ ]:
for cluster in ['all', 1, 2, 3, 4, 5]:
    plt.plot(om[cluster]['decodeWindows'], np.array(om[cluster]['VISall'][80]['imagewise_recall']).mean(axis=1))

In [ ]:
oma['VISall'][80].keys()
plt.plot(oma['decodeWindows'], np.array(oma['VISall'][80]['balanced_accuracy']).mean(axis=1))

In [ ]:
import h5py
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
unitData = h5py.File(active_tensor_file, 'r')

In [ ]:
sessionId = 1091039902
sunits = units.set_index('unit_id').loc[unitData[str(sessionId)]['unitIds'][:]]

In [ ]:
import decoding_utils as du
import vbn_utils

len(vbn_utils.get_unit_ids(sunits.reset_index(), 'VISall', 'all', 'all', clusters=3, clustering='new')), ((du.get_units_in_cluster(sunits, *[3,], clustering='new'))&(du.apply_unit_quality_filter(sunits, no_abnorm=False))&(du.getUnitsInRegion(sunits, 'VISall'))).sum()

## Decoding through the omission

In [ ]:
from importlib import reload

In [ ]:
import vbn_utils

In [ ]:
good_sessions = sessions_table[sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()]['ecephys_session_id'].values

In [ ]:
all_session = []
cluster = 'all'
unitsamplesize =80
for session_id in good_sessions:
    try:
        sess_data = np.load(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding_sliding_window", str(session_id) + f"_throughomissionRS_{cluster}.npy"), allow_pickle=True).item()
        if len(sess_data['VISall'][unitsamplesize]['imagewise_recall'])>0:
            all_session.append(np.array(sess_data['VISall'][unitsamplesize]['imagewise_recall']).mean(axis=1))
    except Exception as e:
        print(f"Error processing session {session_id}: {e}")

In [ ]:
from matplotlib.patches import Rectangle
plt.rcParams['figure.dpi'] = 300

fig, ax = plt.subplots()
fig.suptitle('VIS all, pop size: 80', fontsize=12)
fig.set_size_inches([8, 4])
vbn_utils.mean_sem_plot(np.array(all_session), ax,  x=sess_data['decodeWindows'], color='k')
ax.axhline(0.125, color='k', linestyle='--', linewidth=0.5)
ax.set_xlabel('Time from pre-omission image onset (ms)')
ax.set_ylabel('Decoder accuracy')

filled_rect = Rectangle((0, ax.get_ylim()[1]), 250, 0.05, color='gray', alpha=1, clip_on=False)
ax.add_patch(filled_rect)

open_rect = Rectangle((750, ax.get_ylim()[1]), 250, 0.05, edgecolor='gray', facecolor='none', linewidth=2, clip_on=False)
ax.add_patch(open_rect)

# ax.text(125, ax.get_ylim()[1]+0.02, 'Pre-omission image', ha='center', va='bottom', fontsize=9, color='w')
# ax.text(875, ax.get_ylim()[1]+0.02, 'Omission', ha='center', va='bottom', fontsize=9)
vbn_utils.formatFigure(fig, ax, fontsize=16)